# Entity IDs

## Overview
We will need to join sources on qubit and coupling labels, so a label has to mean the same thing in every source, and keep pointing at the same physical qubit or coupler on the chip over time. If one source numbered them differently, or the control lines were rewired partway through, a "Q5" series could quietly mix two different qubits.

**Question.** Do qubit and coupling IDs mean the same thing in DS1, DS2, DS3 and `output_info`? And does each label stay on the same physical qubit or coupler over time?

**Method.** We parse every source's IDs with `qpu_drift.ids` and compare the sets for qubits (section 1) and couplings (section 2), then look at the odd one out, Q8 (section 3). Last, we test the labels over time against f01, looking for a wiring swap or a chip change (section 4). Results are in [Phase 01, A2](../../docs/phases/01_eda.md#a2-entity-ids).

In [1]:
import numpy as np
import pandas as pd

from qpu_drift.derived.run_map import load_run_entities
from qpu_drift.derived.run_map.clean import ds2_series
from qpu_drift.derived.run_map.constants import DS2_TEMPLATES
from qpu_drift.sources.benchmarking import load_kpi_l2
from qpu_drift.sources.hardware_models import load_values
from qpu_drift.sources.measurements import load_measurements

# Load each source once; every cell below reuses them
ds1 = load_measurements()
kpi = load_kpi_l2()
ent = load_run_entities()
values = load_values()

# DS2 values for the tracked params only
ds2 = ds2_series(values[values["path_template"].isin(DS2_TEMPLATES)])

## 1. DS1, DS2 and DS3 use the same 32 qubits

First, the set of qubit indices each source uses.

In [2]:
# Qubit indices each source uses
qubits = {
    "DS1": set(np.concatenate(ds1["qubits"].dropna().to_numpy()).astype(int)),
    "DS2": set(ds2.loc[ds2["routine"] == "calibrate_1q", "entity"].astype(int)),
    "DS3": set(kpi.loc[kpi["element_type"] == "qubit", "element"].astype(int)),
    "output_info": set(ent.loc[ent["routine"] == "calibrate_1q", "entity"].astype(int)),
}

# How many there are, and which indices of 1-35 are missing
pd.DataFrame(
    {
        src: {"qubits": len(q), "missing from 1-35": sorted(set(range(1, 36)) - q)}
        for src, q in qubits.items()
    }
).T

,qubits,missing from 1-35
DS1,32,"[3, 8, 29]"
DS2,32,"[3, 8, 29]"
DS3,32,"[3, 8, 29]"
output_info,33,"[3, 29]"


DS1, DS2 and DS3 all use 1 to 35 without 3, 8 and 29. `output_info` adds Q8, but only in its first `calibrate_1q` file (section 3).

## 2. Coupling directions agree wherever a coupling is calibrated

Couplings are directed (control-target), so `1-10` and `10-1` are the same physical edge calibrated in opposite directions. We compare the direction sets across sources first, then check which of DS2's directions also have their reverse in DS2.

The sets pool every date, and some couplings switch direction partway through (`periods.BREAKS`, see [A4](a4_windows_breaks.ipynb)) - so both directions of one edge can turn up here without either being wrong.

In [3]:
from qpu_drift.ids import coupling

# Coupling directions per source, written control-target
couplings = {
    "DS1": {
        "-".join(map(str, coupling(c)))
        for c in np.concatenate(ds1["couplings"].dropna().to_numpy())
    },
    "DS2": set(ds2.loc[ds2["routine"] == "calibrate_2q", "entity"]),
    "DS3": set(kpi.loc[kpi["element_type"] == "coupling", "element"]),
    "output_info": set(ent.loc[ent["routine"] == "calibrate_2q", "entity"]),
}

# Size of each set
print({src: len(c) for src, c in couplings.items()})

# Directions that are in one source but not the other
print("output_info, not DS3:", sorted(couplings["output_info"] - couplings["DS3"]))
print("DS3, not output_info:", sorted(couplings["DS3"] - couplings["output_info"]))
print("DS1, not output_info:", sorted(couplings["DS1"] - couplings["output_info"]))
print("output_info, not DS1:", sorted(couplings["output_info"] - couplings["DS1"]))

{'DS1': 36, 'DS2': 68, 'DS3': 35, 'output_info': 39}
output_info, not DS3: ['18-22', '25-26', '35-26', '9-2']
DS3, not output_info: []
DS1, not output_info: ['23-24', '9-12']
output_info, not DS1: ['18-22', '25-17', '25-26', '35-26', '9-2']


In [4]:
# DS2 amp rows per direction, and whether the reverse direction is also in DS2
c2 = ds2[(ds2["routine"] == "calibrate_2q") & (ds2["param"] == "amp")]
d = c2.groupby("entity").size().rename("ds2_rows").to_frame()
d["reverse_in_ds2"] = [f"{b}-{a}" in d.index for a, b in (e.split("-") for e in d.index)]
d["in_output_info"] = d.index.isin(couplings["output_info"])

# Split by presence in output_info and by whether the reverse exists
d.groupby(["in_output_info", "reverse_in_ds2"])["ds2_rows"].agg(
    directions="size", median_rows="median", max_rows="max"
)

directions  median_rows  max_rows
in_output_info reverse_in_ds2                                   
False          False                    2          1.0         1
               True                    27          1.0         1
True           True                    39       1018.0      1101

All 35 DS3 directions are among the 39 that `output_info` calibrates, and so are DS1's apart from `23-24` and `9-12`, which `output_info` never calibrates. DS2 has the same 39, each with about 1,000 amp rows, plus 29 directions with a single row each that never change. So wherever a coupling is actually calibrated, every source writes it the same way round.

## 3. Q8 is a separate site, not an old name for Q30

Q8 is the odd one out, so we look at where it shows up in `output_info` and in the tracked DS2 parameters.

The first file lists Q8 where later files have Q30, so the last cell checks whether Q8 is just an old name for Q30. The cleanest test is the coupling map: a renamed qubit keeps its physical neighbours, so Q8's couplings should match Q30's. As a second check, we take each qubit's f01 in that file (the measured `new` value, else `old`) and set it against the qubit's median over the later runs ($\tilde f_q$ in section 4). If Q8 were Q30, it should sit about as close to Q30's later median as the other qubits sit to their own.

In [5]:
# First 1q file against the later ones: qubits listed and median amp
amp = ent[(ent["routine"] == "calibrate_1q") & (ent["param"] == "amp")]
first = amp["start"].min()
print(
    "first 1q file:", first, "qubits:", sorted(amp.loc[amp["start"] == first, "entity"].astype(int))
)
print(
    f"median amp, first file: {amp.loc[amp['start'] == first, 'new'].median():.3g}"
    f"; later: {amp.loc[amp['start'] > first, 'new'].median():.3g}"
)

# DS2 entries that involve Q8, with first date and number of changes
print("first DS2 snapshot:", values["valid_from_ts"].min())
q8 = ds2[ds2["entity"].astype(str).str.fullmatch(r"8|\d+-8|8-\d+")]
q8.groupby(["routine", "param", "entity"]).agg(
    first=("valid_from_ts", "min"), changes=("valid_from_ts", "size")
)

first 1q file: 2024-03-21 17:03:19 qubits: [1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 31, 32, 33, 34, 35]
median amp, first file: 8e+06; later: 1.36e+07
first DS2 snapshot: 2025-01-01 08:00:13


first  changes
routine      param entity                             
calibrate_2q amp   15-8   2025-01-01 08:00:13        1
                   5-8    2025-01-01 08:00:13        1

In [6]:
# Where Q8 appears in output_info
q8_oi = ent[ent["entity"].astype(str).str.fullmatch(r"8|\d+-8|8-\d+")]
q8_oi.groupby(["routine", "start"])["param"].agg(list)

routine       start              
calibrate_1q  2024-03-21 17:03:19    [freq, amp, drag]
Name: param, dtype: object

In [7]:
# Coupling neighbours of Q8 and Q30 in each source (from the section 2 direction sets)
def neighbours(directions, n):
    return sorted({int(x) for e in directions if str(n) in e.split("-") for x in e.split("-")} - {n})


for n in (8, 30):
    print(f"Q{n} neighbours:", {src: neighbours(c, n) for src, c in couplings.items()})

# Each qubit's f01 in the first 1q file (its new value, else old), in MHz
f0 = ent[(ent["routine"] == "calibrate_1q") & (ent["param"] == "freq")]
f0 = f0[f0["start"] == f0["start"].min()]
f0 = (f0["new"].fillna(f0["old"]) / 1e6).set_axis(f0["entity"].astype(int))

# Each qubit's median f01 over the later runs, using only measured values (as in section 4)
later_f = ent[(ent["routine"] == "calibrate_1q") & (ent["param"] == "freq")]
later_f = later_f[later_f["start"] > later_f["start"].min()]
lq = later_f["entity"].astype(int)
later_meas = later_f.assign(m=later_f["measured"].fillna(False)).pivot_table(
    index="start", columns=lq, values="m", aggfunc="any"
)
later = (
    (later_f.pivot_table(index="start", columns=lq, values="old") / 1e6)
    .where(later_meas.shift(1, fill_value=False).astype(bool))
    .median()
)

# Q8 against Q30, and the qubits whose later level sits closest to Q8
print(f"\nQ8 in the first file: {f0[8]:.1f} MHz; Q30 later: {later[30]:.1f} MHz")
print("closest later levels to Q8 (MHz):", (later - f0[8]).abs().nsmallest(3).round(2).to_dict())

# The other qubits: first-file f01 minus their own later median, and the rank order
d0 = (f0 - later).dropna()
print(
    f"other qubits: {len(d0)}; median difference {d0.median():.2f} MHz; "
    f"|difference| median {d0.abs().median():.2f}, max {d0.abs().max():.2f} MHz"
)
print("largest |difference| (MHz):", d0.abs().nlargest(3).round(2).to_dict())
print(
    f"rank correlation with the later order: {f0[d0.index].rank().corr(later[d0.index].rank()):.3f}"
)

Q8 neighbours: {'DS1': [], 'DS2': [5, 15], 'DS3': [], 'output_info': []}
Q30 neighbours: {'DS1': [31], 'DS2': [31], 'DS3': [31], 'output_info': [31]}



Q8 in the first file: 4404.4 MHz; Q30 later: 4436.1 MHz
closest later levels to Q8 (MHz): {20: 1.93, 34: 3.38, 12: 9.77}
other qubits: 31; median difference 1.05 MHz; |difference| median 3.40, max 92.03 MHz
largest |difference| (MHz): {19: 92.03, 2: 21.78, 25: 7.23}
rank correlation with the later order: 0.973


The first `calibrate_1q` file (2024-03-21) lists Q8 instead of Q30, and its amplitudes are well below later ones (median $8.0 \times 10^{6}$ against $13.6 \times 10^{6}$), so we treat it as a pre-production run. Q8's only other trace is two DS2 entries, `5-8` and `15-8`, which never change. That puts Q8 next to Q5 and Q15, while Q30's only neighbour is Q31 in every source - so Q8 is a different site on the chip.

The f01 check agrees. Q8's first-file value sits 31.7 MHz from Q30's later median, while the other qubits sit a median 3.40 MHz from their own, and only Q19 is further off than Q8 would be.

## 4. Each label stays on the same qubit over time

A label could stay the same while the qubit behind it changes - if two control lines were swapped, or the chip was replaced. Nothing in the register records either, and DS4 only starts in 2025-08 ([A4](a4_windows_breaks.ipynb)), so the test has to come from the values themselves.

We use the `calibrate_1q` freq `old` at each run - the drive frequency in use (leaving out the pre-production first file). It isn't a measurement of the qubit itself, but while the calibration tracks the qubit it sits on f01, so we call it f01 here. A qubit the previous run didn't measure just keeps its last value, so we only keep values the previous run measured. For run $r$ and qubit $q$, let $f_{r,q}$ be that value and $\tilde f_q$ the qubit's median over all runs. The deviation is

$$d_{r,q} = f_{r,q} - \tilde f_q,$$

the chip-wide shift is the median deviation over qubits,

$$s_r = \operatorname*{median}_q \, d_{r,q},$$

and the residual is what's left once that's taken out:

$$\varepsilon_{r,q} = d_{r,q} - s_r.$$

A swap or a chip change should show up as large residuals on many qubits from one run onwards. The qubit order gives another check - the rank correlation between each run's f01 and the medians:

$$\rho_r = \operatorname{corr}\Big(\operatorname{rank}_q f_{r,q}, \; \operatorname{rank}_q \tilde f_q\Big).$$

For each chip-wide episode we also pull out the next run, its shift and any DS2 rollback before it (`periods.UNTRACKED_DS2`). Q23 is the one qubit whose value moves far on its own, so we pull out its level before and after its flagged runs, its measured values before 2025 and its readout history. Then three smaller checks:

- **Register.** `periods.F01_FLAGS` against the flagged windows (a chip-wide shift over 5 MHz, or a qubit residual over 50 MHz outside those). Each window runs from the first flagged run to the first run whose value in use is back within the threshold.
- **Swap.** After a swap, both qubits sit about their f01 gap away from their own median, and stay there. So we take a rolling median of each qubit's $\varepsilon_{r,q}$ over its last 5 measured runs and flag it over 10 MHz (just above the 99th percentile of $|\varepsilon_{r,q}|$). The `periods.INCIDENTS` windows are left out, as the chip-wide episodes already move many qubits at once. We also count how many qubit pairs sit within 10 and 50 MHz of each other.
- **Readout.** How far each qubit's DS2 `R{n}.measure` frequency moves, and how close it sits to the next resonator.

> **Note on limits:** a swap between qubits less than 10 MHz apart stays under the swap check, and swapping neighbours in the order barely moves $\rho_r$. A swap that nobody recalibrated wouldn't move f01 at all.

In [8]:
from qpu_drift import periods

# f01 in MHz: one row per run, one column per qubit
f01 = ent[(ent["routine"] == "calibrate_1q") & (ent["param"] == "freq")]
f01 = f01[f01["start"] > f01["start"].min()]  # drop the pre-production first file
q = f01["entity"].astype(int)
w_all = f01.pivot_table(index="start", columns=q, values="old") / 1e6

# Which qubits each run measured
meas = f01.assign(m=f01["measured"].fillna(False)).pivot_table(
    index="start", columns=q, values="m", aggfunc="any"
)

# Keep only the values the previous run measured
w = w_all.where(meas.shift(1, fill_value=False).astype(bool))
print(
    f"runs: {len(w):,} ({w.index.min():%Y-%m-%d} to {w.index.max():%Y-%m-%d}); qubits: {w.shape[1]}"
)
print(
    "share of runs measuring the qubit, lowest three:", meas.mean().nsmallest(3).round(3).to_dict()
)

# Deviation from each qubit's own median, the chip-wide shift and the residual
dev = w - w.median()
shift = dev.median(axis=1)
res = dev.sub(shift, axis=0)

# Spacing between neighbouring qubits' median f01
spacing = np.diff(np.sort(w.median().dropna().to_numpy()))
print(f"spacing of median f01 (MHz): min {spacing.min():.2f}, median {np.median(spacing):.1f}")

# How large the chip-wide shifts and the residuals get
print(
    "|shift| quantiles (MHz):",
    shift.abs().quantile([0.5, 0.9, 0.99, 0.999, 1]).round(2).to_dict(),
)
print(
    "|residual| quantiles (MHz):",
    res.abs().stack().quantile([0.5, 0.9, 0.99, 0.999, 1]).round(2).to_dict(),
)

# Rank correlation of each run's f01 order with the median order
rank = w.rank(axis=1).corrwith(w.median().rank(), axis=1)
print(
    "rank correlation with the median order:",
    rank.describe()[["min", "25%", "50%"]].round(3).to_dict(),
)
display(rank.nsmallest(5).round(3))

# Qubits with a residual over 50 MHz
big = res.abs() > 50
display(
    pd.DataFrame(
        {
            q: {
                "runs": int(big[q].sum()),
                "first": big.index[big[q]].min(),
                "last": big.index[big[q]].max(),
                "max_MHz": round(res[q].abs().max()),
            }
            for q in big.columns[big.any()]
        }
    ).T
)

# Q23 around its flagged runs: level before and after, and the step per run within
s = w_all[23]
q23 = s[big[23]]
print(
    f"Q23 median f01, 20 runs before / after (MHz): "
    f"{s[s.index < q23.index.min()].tail(20).median():.1f} / "
    f"{s[s.index > q23.index.max()].head(20).median():.1f}; "
    f"median step per run within: {q23.diff().abs().median():.1f}"
)

# Episodes with a chip-wide shift over 5 MHz, the next run, and any DS2 rollback before it
off = shift[shift.abs() > 5]

# Runs more than 2 days apart start a new episode
ep = off.groupby((off.index.to_series().diff() > pd.Timedelta("2D")).cumsum().to_numpy())
episodes = pd.DataFrame(
    {
        "first": ep.apply(lambda s: s.index.min()),
        "last": ep.apply(lambda s: s.index.max()),
        "runs": ep.size(),
        "min_shift": ep.min().round(1),
    }
)

# The next run after each episode, its shift, and any DS2 rollback in between
episodes["next_run"] = [w.index[w.index > t][0] for t in episodes["last"]]
episodes["next_run_shift"] = shift[episodes["next_run"]].round(1).to_numpy()
rollbacks = [u.at for u in periods.UNTRACKED_DS2 if u.kind == "rollback"]
episodes["rollback_before_next_run"] = [
    max((r for r in rollbacks if a < r < b), default=None)
    for a, b in zip(episodes["last"], episodes["next_run"])
]
episodes

runs: 1,297 (2024-04-12 to 2026-08-18); qubits: 32
share of runs measuring the qubit, lowest three: {2: 0.003, 13: 0.003, 23: 0.791}
spacing of median f01 (MHz): min 0.04, median 5.9
|shift| quantiles (MHz): {0.5: 0.01, 0.9: 1.34, 0.99: 2.2, 0.999: 57.83, 1.0: 59.29}
|residual| quantiles (MHz): {0.5: 0.05, 0.9: 3.52, 0.99: 7.38, 0.999: 301.01, 1.0: 753.99}
rank correlation with the median order: {'min': 0.824, '25%': 0.997, '50%': 0.998}


start
2026-07-05 19:34:00    0.824
2026-07-05 08:45:41    0.851
2026-07-04 19:35:39    0.855
2026-07-04 08:45:41    0.878
2026-07-03 19:39:59    0.889
dtype: float64

,runs,first,last,max_MHz
23,56,2025-03-24 19:44:14,2025-04-24 08:45:21,754
28,2,2026-07-05 08:45:41,2026-07-05 19:34:00,60
30,2,2026-07-05 08:45:41,2026-07-05 19:34:00,53
32,2,2026-07-05 08:45:41,2026-07-05 19:34:00,58
34,2,2026-07-05 08:45:41,2026-07-05 19:34:00,56


Q23 median f01, 20 runs before / after (MHz): 4303.0 / 4287.5; median step per run within: 11.1


,first,last,runs,min_shift,next_run,next_run_shift,rollback_before_next_run
0,2026-03-09 19:42:45,2026-03-11 19:43:57,5,-59.3,2026-03-24 14:45:52,-2.3,2026-03-24 12:03:12
1,2026-07-02 19:38:25,2026-07-05 19:34:00,7,-58.2,2026-07-05 21:54:09,-2.2,2026-07-05 21:25:30


Over 1,297 runs the qubit order barely moves (median $\rho_r$ 0.998), and only two stretches shift the whole chip by more than 5 MHz: 2026-03-09 to 03-11 and 2026-07-02 to 07-05, both by about −58 MHz. Each ends with a DS2 rollback just before the next run, so that run shows the restored values - it tells us the bad values were undone, not that the qubits came back. Q2 and Q13 are measured in only 4 runs each, so these checks barely cover them.

Q23 is the one qubit that moves far on its own: from 2025-03-24 to 04-24 its value sits up to 754 MHz from its median. That isn't the qubit moving. No other qubit gets past 60 MHz, and Q23's value walks a median 11.1 MHz per run, so the calibration has lost the qubit and each run's fit pushes the drive frequency further off. What might be physical is the change in level across it, from 4303.0 MHz over the 20 runs before to 4287.5 MHz over the 20 after.

Next, we check these windows against `periods.F01_FLAGS`.

In [9]:
from itertools import zip_longest

# Register check: periods.F01_FLAGS against the windows above.
# Each window ends at the first run whose value in use is back within the threshold.
res_all = (w_all - w.median()).sub(shift, axis=0)  # every value in use, measured or not

# First run after t where the condition holds again
back = lambda ok, t: ok.index[(ok.index > t) & ok.to_numpy()][0]

# Chip-wide windows: first flagged run to the first run back within 5 MHz
chip = [(e["first"], back(shift.abs() <= 5, e["last"])) for _, e in episodes.iterrows()]

# Per-qubit windows: first flagged run to the first run back within 50 MHz
qubit = {
    q: (big.index[big[q]].min(), back(res_all[q].abs() <= 50, big.index[big[q]].max()))
    for q in big.columns[big.any()]
}

# A qubit window inside a chip-wide window is not a separate flag
inside = {q: any(a <= s and e <= b for a, b in chip) for q, (s, e) in qubit.items()}
print("qubit windows inside a chip-wide window:", [q for q, v in inside.items() if v])

# Computed windows next to the register entries
computed = sorted(
    [((s, e), (q,)) for q, (s, e) in qubit.items() if not inside[q]] + [(p, None) for p in chip],
    key=lambda c: c[0][0],
)
register = [(f.period, f.qubits) for f in periods.F01_FLAGS]
pd.DataFrame(zip_longest(register, computed), columns=["register", "computed"]).assign(
    match=lambda d: d["register"] == d["computed"]
)

qubit windows inside a chip-wide window: [28, 30, 32, 34]


,register,computed,match
0,"((2025-03-24 19:44:14, 2025-04-25 12:13:42), (...","((2025-03-24 19:44:14, 2025-04-25 12:13:42), (...",True
1,"((2026-03-09 19:42:45, 2026-03-24 14:45:52), N...","((2026-03-09 19:42:45, 2026-03-24 14:45:52), N...",True
2,"((2026-07-02 19:38:25, 2026-07-05 21:54:09), N...","((2026-07-02 19:38:25, 2026-07-05 21:54:09), N...",True


All three `F01_FLAGS` entries match. The four qubits over 50 MHz on 2026-07-05 (Q28, Q30, Q32 and Q34) fall inside the second chip-wide window, so they don't need flags of their own.

Then the swap check.

In [10]:
# Swap check. How many measured values each qubit has, lowest four
print("measured values per qubit, lowest four:", w.notna().sum().nsmallest(4).to_dict())

# Qubit pairs by the gap between their median f01
med = w.median().dropna()
gaps = pd.Series(
    {(a, b): abs(med[a] - med[b]) for i, a in enumerate(med.index) for b in med.index[i + 1 :]}
)
print("closest pairs (MHz):", gaps.nsmallest(3).round(2).to_dict())
print(f"pairs within 10 / 50 MHz: {(gaps < 10).sum()} / {(gaps < 50).sum()} of {len(gaps)}")

# Rolling median of each qubit's residual over its last 5 measured runs, INCIDENTS left out
calm = res[~periods.within(res.index, *(i.period for i in periods.INCIDENTS))]
roll = calm.apply(lambda s: s.dropna().rolling(5).median().reindex(s.index))
print(
    "|rolling median residual| quantiles (MHz):",
    roll.abs().stack().quantile([0.5, 0.99, 0.999, 1]).round(2).to_dict(),
)

# Q23: its measured values before 2025 (they open its first 5-run window), and its overall median
s23 = w[23].dropna()
early = s23[s23.index < "2025"]
print(
    f"Q23: {len(early)} measured values before 2025 ({early.index.min():%Y-%m-%d} to "
    f"{early.index.max():%Y-%m-%d}, median {early.median():.1f} MHz); "
    f"first from 2025: {s23[s23.index >= '2025'].index.min()}; median over all runs {s23.median():.1f} MHz"
)

# Qubits whose rolling median passes 10 MHz
step = roll.abs() > 10
pd.DataFrame(
    {
        q: {
            "runs": int(step[q].sum()),
            "first": step.index[step[q]].min(),
            "last": step.index[step[q]].max(),
            "max_MHz": round(roll[q].abs().max(), 1),
        }
        for q in step.columns[step.any()]
    }
).T

measured values per qubit, lowest four: {2: 4, 13: 4, 23: 1025, 35: 1256}
closest pairs (MHz): {(5, 35): 0.04, (22, 25): 0.08, (5, 11): 0.96}
pairs within 10 / 50 MHz: 32 / 172 of 496


|rolling median residual| quantiles (MHz): {0.5: 0.05, 0.99: 7.33, 0.999: 303.95, 1.0: 726.51}
Q23: 4 measured values before 2025 (2024-04-17 to 2024-04-23, median 4304.1 MHz); first from 2025: 2025-02-13 08:00:21; median over all runs 4287.6 MHz


,runs,first,last,max_MHz
23,115,2025-02-13 08:00:21,2025-04-26 08:45:21,726.5
30,3,2024-05-13 23:15:05,2024-05-16 23:15:05,10.8


Only Q23 and Q30 pass 10 MHz, and neither has a partner moving with it, so there's no sign of a swap between qubits more than 10 MHz apart. Q30's flag is three runs in 2024-05, just over the line (10.8 MHz). Q23's starts at its first 2025 value (2025-02-13), well before the excursion. That's because its median over all runs (4287.6 MHz) matches its level after the excursion, so its level before already reads about 15 MHz high. The check can't see swaps between close qubits, though: 32 of the 496 pairs are within 10 MHz of each other.

Last, the readout frequencies, which would move with a swap even where f01 doesn't.

In [11]:
# DS2 readout frequencies of each qubit
ro = values[values["path_template"] == "model.pulse_channels{Ch*.R*.measure}.pulse_channel.frequency"]
ro = (
    ro[["valid_from_ts", "value_float"]]
    .join(ro["path"].str.extract(r"\{Ch\d+\.R(?P<qubit>\d+)\.measure\}"))
    .dropna(subset=["qubit"])
)

# Per-qubit range in MHz, and the gap to the next resonator up
band = (
    ro.groupby(ro["qubit"].astype(int))["value_float"]
    .agg(["min", "max", "size"])
    .div([1e6, 1e6, 1])
    .sort_values("min")
)
band["range_MHz"] = band["max"] - band["min"]
band["next_qubit"] = band.index.to_series().shift(-1).astype("Int64")
band["gap_to_next_MHz"] = band["min"].shift(-1) - band["max"]
print(
    f"readout: {len(band)} qubits, "
    f"{ro['valid_from_ts'].min():%Y-%m-%d} to {ro['valid_from_ts'].max():%Y-%m-%d}; "
    f"values per qubit {band['size'].min():.0f}-{band['size'].max():.0f}; "
    f"largest range {band['range_MHz'].max():.2f} MHz"
)

# Q23's readout frequency at each DS2 change, around its f01 excursion
r23 = ro[ro["qubit"] == "23"].sort_values("valid_from_ts")
print(
    "Q23 readout (MHz):",
    dict(zip(r23["valid_from_ts"].dt.strftime("%Y-%m-%d %H:%M"), (r23["value_float"] / 1e6).round(2))),
)

# Resonators within 1 MHz of the next one up
band[band["gap_to_next_MHz"] < 1].round(2)

readout: 32 qubits, 2025-01-01 to 2026-03-30; values per qubit 3-7; largest range 1.16 MHz
Q23 readout (MHz): {'2025-01-01 08:00': 10261.58, '2025-02-12 13:40': 10262.32, '2025-02-12 19:05': 10262.32, '2025-04-25 09:58': 10261.97, '2026-03-24 14:45': 10262.19}


,min,max,size,range_MHz,next_qubit,gap_to_next_MHz
qubit,,,,,,
14,9627.85,9627.93,4.0,0.08,6,0.64
6,9628.57,9629.72,5.0,1.16,19,-0.58
9,9832.61,9832.66,4.0,0.05,5,0.47
16,9839.39,9839.74,7.0,0.35,21,0.19
27,9845.18,9845.26,3.0,0.08,11,0.07
15,10053.17,10054.17,3.0,1.01,4,-0.75
12,10054.88,10056.04,4.0,1.15,1,-0.42
17,10261.20,10261.30,3.0,0.10,23,0.28


No qubit's readout frequency moves by more than 1.16 MHz, but DS2 only has them from 2025-01-01 to 2026-03-30, so they miss the July 2026 stretch. Q23's readout stays between 10261.58 and 10262.32 MHz through its excursion. It changes once on 2025-04-25 09:58, just before the first run back in range (12:13), which suggests the qubit was recalibrated by hand. Eight resonators sit within 1 MHz of the next one up (Q17 is 0.28 MHz below Q23), so readout alone can't rule out a swap between those pairs.

## Conclusion

| Result | Status |
|---|---|
| DS1, DS2 and DS3 use the same 32 qubits; `output_info` adds Q8 in its pre-production first file only | Confirmed |
| Couplings are directed, and every source writes a calibrated coupling the same way round | Confirmed |
| Q8 is a separate site, not an old name for Q30: its neighbours are Q5 and Q15, while Q30's only neighbour is Q31 | Confirmed |
| No sign of a new chip or of a recalibrated swap between qubits over 10 MHz apart: the qubit order holds and no two qubits move off together. Q2 and Q13 rest on the readout check alone | Confirmed |
| Q23 keeps its label through 2025-03 and 04. Its value moving up to 754 MHz away is the calibration losing the qubit, not the qubit moving; its 15 MHz level change across that stretch is unexplained | Confirmed (level change open) |
| The two chip-wide shifts of about −58 MHz (2026-03 and 2026-07) aren't a chip change: every later run is back within 5 MHz of the usual level | Confirmed |
| `periods.F01_FLAGS` matches the three computed windows | Confirmed |